# Recession Impact Analysis
## Data Collection & Cleaning

Downloads official U.S. employment data from FRED, cleans it, and saves one master table.

---


### Step 1: Load tools

Python libraries for data handling, date alignment, and FRED API connection.


In [7]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from fredapi import Fred

### Step 2: Connect to FRED

You need a free FRED API key. If you do not have one, sign up here:
https://fred.stlouisfed.org/docs/api/api_key.html

Paste your key into the `.env` file in the project folder as:
`FRED_API_KEY=your_actual_key_here`


In [8]:
load_dotenv()
FRED_API_KEY = os.getenv("FRED_API_KEY")

if not FRED_API_KEY or FRED_API_KEY == "your_fred_api_key_here":
    raise ValueError("Please set your FRED_API_KEY in the .env file before running this notebook")

fred = Fred(api_key=FRED_API_KEY)
print("FRED connection ready")

FRED connection ready


### Step 3: Download all 6 FRED series fresh

Raw series fetched directly from FRED with no cached CSV merge.


In [9]:
tickers = {
    "USREC": "recession_indicator",
    "PAYEMS": "total_nonfarm_payroll",
    "USCONS": "construction_employment",
    "MANEMP": "manufacturing_employment",
    "USPBS": "professional_business_services",
    "USINFO": "information_sector",
}

raw_series = {}
for fred_id, col_name in tickers.items():
    print(f"Downloading {fred_id}...")
    s = fred.get_series(fred_id)
    raw_series[col_name] = s
    print(f"   {len(s)} observations from {s.index.min().date()} to {s.index.max().date()}")

print("\nAll 6 series downloaded")

   2063 observations from 1854-12-01 to 2026-10-01
   1054 observations from 1939-01-01 to 2026-10-01
   1054 observations from 1939-01-01 to 2026-10-01
   1054 observations from 1939-01-01 to 2026-10-01
   1054 observations from 1939-01-01 to 2026-10-01
   1054 observations from 1939-01-01 to 2026-10-01

All 6 series downloaded


### Step 4: Find overlapping start date

Determine the first date where ALL 6 series have data. This avoids pre-1939 rows
where only USREC exists and employment columns are empty.


In [10]:
first_dates = {}
for col_name, series in raw_series.items():
    first_dates[col_name] = series.index.min()
    print(f"{col_name}: starts {series.index.min().date()}")

overlapping_start_date = max(first_dates.values())
print(f"\nActual overlapping start date: {overlapping_start_date.date()}")
print(f"All 6 tickers have data from this date forward.")

recession_indicator: starts 1854-12-01
total_nonfarm_payroll: starts 1939-01-01
construction_employment: starts 1939-01-01
manufacturing_employment: starts 1939-01-01
professional_business_services: starts 1939-01-01
information_sector: starts 1939-01-01

Actual overlapping start date: 1939-01-01
All 6 tickers have data from this date forward.


### Step 5: Clean and align to monthly frequency

Resample each series to month-end, forward-fill short gaps, then slice to the
overlapping start date so all 6 columns have complete overlapping data.


In [11]:
cleaned = {}
for col_name, series in raw_series.items():
    monthly = series.resample("ME").last()
    monthly = monthly.ffill()
    cleaned[col_name] = monthly.loc[overlapping_start_date:]

print(f"All series cleaned to monthly frequency from {overlapping_start_date.date()}")
print(f"Rows per series: {len(cleaned[list(cleaned.keys())[0]])}")

All series cleaned to monthly frequency from 1939-01-01
Rows per series: 1054


### Step 6: Merge into master table

Stack all 6 raw series side by side into one table indexed by date.


In [1]:
master = pd.concat(cleaned, axis=1) master.index.name = "date"\nmaster = master.dropna(how="any")\n\nprint(f"Master table shape: {master.shape}")\nprint(f"Date range: {master.index.min().date()} to {master.index.max().date()}")\nprint(f"Columns: {list(master.columns)}")\nprint(f"Any NaN remaining: {master.isna().any().any()}")

SyntaxError: invalid syntax (669007383.py, line 1)

### Step 7: Feature engineering

Create blue-collar and white-collar job aggregates plus their percentage shares
of the core cohort.


In [ ]:
master["blue_collar_jobs"] = master["construction_employment"] + master["manufacturing_employment"]\nmaster["white_collar_jobs"] = master["professional_business_services"] + master["information_sector"]\nmaster["core_cohort_total"] = master["blue_collar_jobs"] + master["white_collar_jobs"]\nmaster["white_collar_pct"] = (master["white_collar_jobs"] / master["core_cohort_total"]) * 100\nmaster["blue_collar_pct"] = (master["blue_collar_jobs"] / master["core_cohort_total"]) * 100\n\nprint("Feature engineering complete")\nprint(f"New columns: blue_collar_jobs, white_collar_jobs, core_cohort_total, white_collar_pct, blue_collar_pct")\nprint(master[['blue_collar_jobs', 'white_collar_jobs', 'white_collar_pct', 'blue_collar_pct']].head())

### Step 8: Save master table

Saves the clean dataframe to `data/processed/master_table.csv` with automatic
folder creation if the directory does not exist.


In [ ]:
out_path = os.path.join("..", "data", "processed", "master_table.csv")\nabs_path = os.path.abspath(out_path)\nprint(f"Attempting to save to: {abs_path}")\nos.makedirs(os.path.dirname(abs_path), exist_ok=True)\nprint(f"Directory created/exists: {os.path.dirname(abs_path)}")\nmaster.to_csv(abs_path)\n\nprint(f"\\nSaved master table to {abs_path}")\nprint(f"Shape: {master.shape[0]} rows x {master.shape[1]} columns")\nprint(f"Date range: {master.index.min().date()} to {master.index.max().date()}")\nprint(master.head(10))

### Step 9: Final verification

Confirms the saved file is readable and aligned for Notebooks 2 and 3.


In [ ]:
verify = pd.read_csv(abs_path, parse_dates=["date"], index_col="date")\nprint("Verification of saved CSV:")\nverify.info()\nprint(f"\\nFirst 5 rows:")\nprint(verify.head())\nprint(f"\\nLast 5 rows:")\nprint(verify.tail())\nprint(f"\\nColumn check:")\nprint(f"Expected columns: recession_indicator, total_nonfarm_payroll, construction_employment, manufacturing_employment, professional_business_services, information_sector, blue_collar_jobs, white_collar_jobs, core_cohort_total, white_collar_pct, blue_collar_pct")\nprint(f"Actual columns: {list(verify.columns)}")\nprint(f"All present: {set(['recession_indicator', 'total_nonfarm_payroll', 'construction_employment', 'manufacturing_employment', 'professional_business_services', 'information_sector', 'blue_collar_jobs', 'white_collar_jobs', 'core_cohort_total', 'white_collar_pct', 'blue_collar_pct']).issubset(set(verify.columns))}")